In [ ]:
import pandas as pd
import numpy as np
import get_state
import datetime
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

class Combination:
    def __init__(self, series: pd.Series, num_bins: int = 4):
        thresholds = pd.Series(np.linspace(-100, 100, num=num_bins))
        self.combination = np.digitize(series, thresholds, right=False)

    def as_vector(self):
        return self.combination

def transition_knn(year_horizon: int = 2015,
                   col: str = 'Open_adjusted',
                   days: int = 3,
                   num_bins: int = 4,
                   k: int = 5):
    df = pd.read_csv('inflation_adjusted_berkshire_stocks.csv')
    df['Date'] = pd.to_datetime(df['Date'])
    df = df[df['Date'] > datetime.datetime(year_horizon, 1, 1)]

    data_list = [df[col].iloc[i:i+days].reset_index(drop=True) for i in range(len(df) - days)]
    state_vectors = [Combination(state, num_bins=num_bins).as_vector() for state in data_list]
    X = np.array(state_vectors[:-1])
    y_raw = [str(state) for state in state_vectors[1:]]
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)

    current_state = state_vectors[-1].reshape(1, -1)
    predicted_label = knn.predict(current_state)
    predicted_proba = knn.predict_proba(current_state)
    predicted_state = label_encoder.inverse_transform(predicted_label)[0]

    print("Predicted next state:", predicted_state)
    print("Prediction probabilities:")
    for idx, prob in enumerate(predicted_proba[0]):
        state_str = label_encoder.inverse_transform([idx])[0]
        print(f"State {state_str}: Probability {prob}")

    return knn, label_encoder

transition_knn()